# 酒店内容推荐系统复刻

* 数据库：西雅图酒店数据集
    * 下载地址：https://github.com/susanli2016/Machine-Learning-with-Python/blob/master/Seattle_Hotels.csv
    * 字段：name, address, desc

* 任务：基于用户选择的酒店，推荐相似度高的Top10其他酒店
* 方法：计算当前酒店特征向量与整个酒店特征矩阵的余弦相似度，取相似度最大的Top-k个
* 整体流程：清洗后的酒店文案 → 切分出 1~3 词长短语 → 删掉英文无意义停用词 → 计算每个词组 TF-IDF 权重 → 生成文本特征稀疏矩阵

In [35]:
# 导入库以及依赖
import pandas as pd
import matplotlib.pyplot as plt
# sklearn 文本向量化工具，用来切分文本、分词、统计词语出现次数
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [36]:
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用于正常显示中文

In [37]:
# 导入酒店信息csv表格
df = pd.read_csv('Seattle_Hotels.csv', encoding="latin-1")

In [38]:
# 初步查看表格的情况
print(df.head())
print("===================================")
print(df.info())
print("===================================")
print(df.describe())
print("===================================")

                             name  ...                                               desc
0  Hilton Garden Seattle Downtown  ...  Located on the southern tip of Lake Union, the...
1          Sheraton Grand Seattle  ...  Located in the city's vibrant core, the Sherat...
2   Crowne Plaza Seattle Downtown  ...  Located in the heart of downtown Seattle, the ...
3   Kimpton Hotel Monaco Seattle   ...  What?s near our hotel downtown Seattle locatio...
4              The Westin Seattle  ...  Situated amid incredible shopping and iconic a...

[5 rows x 3 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   name     152 non-null    object
 1   address  152 non-null    object
 2   desc     152 non-null    object
dtypes: object(3)
memory usage: 3.7+ KB
None
                                  name  ...                                               desc
count    

## 1. ngram_range 基础语法
`CountVectorizer` 里 `ngram_range=(min_gram, max_gram)`
- 第一个数：**最小词组包含几个单词**
- 第二个数：**最大词组包含几个单词**
程序会提取「单词数量介于 min ~ max 之间」所有连续词组。
* 举例直观对比

拿一句话举例文本：`the room is clean` 

① `ngram_range=(1,1)` 也就是你代码里 `(n,n)` 当 n=1

只拆分**单个单词**：
`the`、`room`、`is`、`clean`
**只留 1 个词的单元，不会出现 2 个词、3 个词的组合**

② `ngram_range=(2,2)` 也就是 n=2

只拆分**连续两个单词**：
`the room`、`room is`、`is clean`
**只提取两两组合，绝对不会混入单个词、三个词**

③ 如果写成 `ngram_range=(1,2)`（不是固定 n）

会同时提取 1 词 + 2 词 两种粒度：
单个词：`the` `room` `is` `clean`
两词短语：`the room` `room is` `is clean`
✅ 这就是**混合长短词组**，1 词和 2 词混在一起统计。

* 为什么 `(n, n)` 就是固定长度、不混合

因为最小值和最大值完全相等：
`最小长度 = 最大长度 = n`
分词器没有中间可选的长度，只能切出恰好 n 个连续单词构成的短语，没有别的尺寸，自然不会混杂不同长度的词组。

* 对应你函数的设计目的

函数参数只传一个 n，强制写成 `ngram_range=(n,n)`，就是**强制锁定分词粒度**：

- 想统计单个高频词 → n=1
- 想统计双词搭配短语 → n=2
- 想统计三词固定短句 → n=3
每一次调用只分析一种长度的文本单元，避免不同维度数据混在一起，词频对比才有意义。

## get_top_n_words入参说明

1. `corpus`：语料文本列表 / Series，这里传 `df['desc']`，就是整张表所有酒店描述文本
2. `n=1`：代表**N-gram**的 n 值
   - `n=1`：一元分词（单个单词，比如 `room`、`clean`）
   - `n=2`：二元词组（两个连续单词，比如 `very clean`）
   - `n=3`：三元短语
3. `k=None`：最终返回词频排名前 k 个词组，不传就返回全部

CountVectorizer参数解析：
1. `ngram_range=(n, n)`：只提取固定长度 n 的 gram，不会混合长短词组
2. `stop_words='english'`：加载英文内置停用词表，自动过滤无意义虚词
比如 `a/an/the/and/of/is/in/on` 这类助词、介词，不参与统计
3. `.fit(corpus)`：遍历全部酒店描述文本，生成词表词典，学习分词规则

bag_of_word：
- `transform`：把原始文本转为**词袋稀疏矩阵** 
词袋稀疏矩阵核心思想：**抛弃句子语序、语法，只统计文本里「单词有没有出现、出现多少次」**，把一句话当成一堆单词的袋子，只计数不关心顺序。
- 矩阵维度：`行数=文本条数，列数=总词汇数量`
- 矩阵里每个数值 = 该条文本中对应单词出现次数

1. 普通稠密矩阵：会把所有 0 也存进内存，极度耗内存，极易卡顿崩溃
2. **稀疏矩阵**：只存储「非 0 的位置 + 对应数值」，所有 0 直接忽略不保存，极大节省存储空间。
`CountVectorizer` 默认输出就是 **scipy 稀疏矩阵（csr 格式）**，也就是你代码里的 `bag_of_words`。

## 1. fit (corpus)：学习词典，只做「建表」，不生成矩阵

```
vec = CountVectorizer(...).fit(corpus)
```

遍历全部输入文本语料，完成两件事：

1. 英文分词、按 `ngram_range` 切词；
2. 过滤停用词（a/an/the/and 等）；
3. 生成**词汇词典 `vec.vocabulary_`**：`{单词/短语 : 它在矩阵里对应的列下标}`。

> 
> 这一步只是**定好列的顺序**，相当于规定好每一列代表哪个词，还没有生成任何计数矩阵。

## 2. transform (corpus)：根据建好的词典，把文本批量转稀疏词袋矩阵

```
bag_of_words = vec.transform(corpus)
```

逐行处理每一条文本，对照前面 fit 出来的词典：

1. 对单条文本分词；
2. 看每个词在词典里对应第几列；
3. 在该文本对应行、该列位置 +1 计数；
4. 所有没出现的单词全部记为 0，稀疏存储不占用空间。

最终结果：

- 行数 = 输入 `corpus` 有多少条酒店描述
- 列数 = 总词汇数量
- 数据结构：CSR 稀疏矩阵，只记录非零值

## 补充：fit\_transform () 简写

`vec.fit_transform(corpus)`
等价于先 `fit` 建词典，立刻接着 `transform` 转矩阵，合并成一步。
你代码里分开写 fit + transform 是标准写法。

## sum
- `axis=0`：按**列求和**
- 每一列代表一个单词，把这一列所有行数字全部相加
- 结果就是**该单词在所有酒店描述全文本里的总出现频次**，后续用来排序取 Top 高频词。

In [39]:
# 得到酒店描述中N-gram特征中的Topk个
def get_top_n_words(corpus, n=1, k=None):
    # 统计n-gram的词频矩阵
    # from sklearn.feature_extraction.text import CountVectorizer
    # `CountVectorizer`：sklearn 文本向量化工具，用来切分文本、分词、统计词语出现次数
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    # 按照词频从大到小排序
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:k]

common_words1 = get_top_n_words(df['desc'], 1, 20)
df1 = pd.DataFrame(common_words1, columns=['desc', 'count'])
df1.groupby('desc').sum()['count'].sort_values().plot(kind='barh', title='去掉停用词后酒店描述中的Top20单词')
plt.show()

In [40]:
# Bi-Gram
common_words2 = get_top_n_words(df['desc'], 2, 20)
df2 = pd.DataFrame(common_words2, columns=['desc', 'count'])
df2.groupby('desc').sum()['count'].sort_values().plot(kind='barh', title='去掉停用词后酒店描述中的Top20单词')
plt.show()

In [41]:
# Tri-Gram
common_words3 = get_top_n_words(df['desc'], 3, 20)
df3 = pd.DataFrame(common_words3, columns=['desc', 'count'])
df3.groupby('desc').sum()['count'].sort_values().plot(kind='barh', title='去掉停用词后酒店描述中的Top20单词')
plt.show()

In [42]:
def clean_text(text):
    # 全部转小写
    text = text.lower()
    return text

In [43]:
df['desc']  # 酒店描述可以观察到有大写有小写

0      Located on the southern tip of Lake Union, the...
1      Located in the city's vibrant core, the Sherat...
2      Located in the heart of downtown Seattle, the ...
3      What?s near our hotel downtown Seattle locatio...
4      Situated amid incredible shopping and iconic a...
                             ...                        
147    Located in Queen Anne district, The Halcyon Su...
148    Just a block from the world famous Space Needl...
149    Stay Alfred on Wall Street resides in the hear...
150    The perfect marriage of heightened convenience...
151    Yes, it's true. Every room at citizenM is the ...
Name: desc, Length: 152, dtype: object

In [44]:
df['desc_clean'] = df['desc'].apply(clean_text)

In [45]:
df['desc_clean']  # 酒店描述已经成功转为全部小写

0      located on the southern tip of lake union, the...
1      located in the city's vibrant core, the sherat...
2      located in the heart of downtown seattle, the ...
3      what?s near our hotel downtown seattle locatio...
4      situated amid incredible shopping and iconic a...
                             ...                        
147    located in queen anne district, the halcyon su...
148    just a block from the world famous space needl...
149    stay alfred on wall street resides in the hear...
150    the perfect marriage of heightened convenience...
151    yes, it's true. every room at citizenm is the ...
Name: desc_clean, Length: 152, dtype: object

`TfidfVectorizer` 是 sklearn 封装好的**TF-IDF 向量化工具**，用来把英文文本转换成 TF-IDF 数值特征矩阵，是 NLP 最经典的文本特征提取方式。

## 扩展：NLP
TF-IDF、CountVectorizer 词袋、N-gram 全都属于 **NLP 文本表示（文本向量化）** 范畴。
核心目的：人类语言文字无法直接喂给机器学习模型，必须把文字转换成数字向量，这个过程叫**文本特征工程**，是 NLP 最基础前置步骤。

适用场景：文本分类、情感分析、文本聚类、文本相似度计算、关键词提取等酒店评论分析场景。
### NLP 文本向量化经典方法（按发展顺序划分四大阶段）

#### 阶段 1：传统统计方法（机器学习时代，你现在在用的）

无需深度学习，基于词频统计，简单高效、训练快、适合小数据集，就是你目前学的这套：

##### 1. 词袋模型 BoW（CountVectorizer）

只统计单词出现次数，完全忽略语序、上下文语义。
缺点：

- 不区分语序：`cat bites dog` 和 `dog bites cat` 向量一模一样
- 高频通用词权重过大

##### 2. N-gram 扩充词袋

在单词基础上加入连续词组（2 词、3 词短语），轻微弥补语序缺失，你代码里 `ngram_range=(1,3)` 就是这个。

##### 3. TF-IDF（TfidfVectorizer）【你代码正在使用】

在词袋基础上做全局权重惩罚：
词到处都出现 → 权重变低；小众特色词汇 → 权重拉高
优点：效果远好于单纯词袋，工业传统项目大量使用
缺点：依旧没有语义理解，词语之间不存在关联（hotel、inn、lodging 同义，但向量完全无关）

##### 4. 哈希词袋 HashingVectorizer

不存储完整词汇表，用哈希映射压缩维度，适合超大规模海量文本，极少用在常规数据分析。

---

#### 阶段 2：分布式表示（词向量 Word Embedding，深度学习入门）

核心思想：给每个单词学习一个固定长度稠密向量，**语义相近的单词，向量空间距离很近**，解决 TF-IDF 无语义的痛点。

##### 1. Word2Vec（最经典）

分两种训练模式：

- CBOW：用上下文预测中间单词
- Skip-Gram：用中心单词预测周边上下文
产出：每个单词对应 50/100/300 维向量
局限：**一词多义无法区分**，bank（银行 / 河岸）共用同一个向量；只能得到单词向量，不能直接得到整句话向量。

##### 2. GloVe

融合全局词频统计 + Word2Vec 上下文，兼顾统计信息与语义，效果略优于 Word2Vec。

##### 3. FastText

Facebook 推出，支持英文词根、前缀后缀，生僻词也能生成向量；训练速度极快，短文本、评论场景很适配酒店数据集。

> 
> 整句文本做法：把句子里所有单词向量相加 / 求平均，得到句子整体向量。

---

#### 阶段 3：句子级向量模型（直接输出整段文本向量）

不需要自己拼接词向量，开箱即用输出整条酒店描述的向量：

##### 1. Doc2Vec

Word2Vec 升级版，加入段落向量标识，训练时同时学习单词 + 文档向量，专门用于长文本向量化。

##### 2. Sentence-BERT / BERT 句向量

基于预训练大模型微调，专门优化**句子相似度任务**，酒店评论比对、聚类效果碾压 TF-IDF。

---

#### 阶段 4：预训练语言大模型（当下主流 SOTA，深度 NLP）

基于 Transformer 架构，上下文双向理解，一词多义可区分，能捕捉语境、语法、情感，是现在工业界标配：

##### 1. BERT 系列（开山之作）

双向编码器，根据前后全文语境动态生成词向量：同一个单词在不同句子里向量不一样。
衍生轻量化版本：DistilBERT、MobileBERT，低配电脑也能运行。

##### 2. RoBERTa、ALBERT

BERT 优化版，去掉冗余参数、加大训练语料，文本分类 / 聚类精度更高。

##### 3. 通用中文 / 英文大模型

- 英文：GPT、DistilGPT2、DeBERTa
- 中文：ERNIE、MacBERT、ChatGLM 嵌入层
优势：
不仅能提取文本特征，还可直接做情感判断、关键词抽取、文本摘要、翻译等复杂 NLP 任务。

##### 4. 多语言模型 LaBSE

支持中英双语文本向量对齐，如果你后续要做多语言酒店评价分析非常合适。

### 各方法优劣对比（结合你的酒店描述数据分析场景）


| 方法 | 优点 | 缺点 | 适用场景 |
| --- | --- | --- | --- |
| TF-IDF+Ngram | 速度快、无 GPU 也能跑、易解释、小数据集稳定 | 无语义、维度极高 | 入门练习、快速 baseline 基线模型 |
| Word2Vec/FastText | 具备基础语义，向量维度小 | 静态词向量，无法区分一词多义 | 中等文本聚类 |
| Sentence-BERT | 语义理解极强，相似度精准 | 需要一点算力，速度偏慢 | 酒店评论相似度、精品聚类、情感分析 |
| BERT 类预训练模型 | NLP 效果天花板，适配所有下游任务 | 算力要求最高 | 高精度数据分析、竞赛项目 |

## TfidfVectorizer参数解析——初始化 TF-IDF 转换器
1. `analyzer='word'`

分词粒度：按**英文单词**拆分文本（默认就是 word）
可选：`char` 按单个字符切分，这里酒店描述文本用单词分词最合适。

2. `ngram_range=(1, 3)`

同时提取三种长度的连续词组：
1-gram（单个单词）、2-gram（两个连续单词短语）、3-gram（三个单词短句）
示例文本：`room very clean`
拆分出所有单元：
单个词：room, very, clean
双词：room very, very clean
三词：room very clean
最终这些全部都会作为特征列，一并纳入 TF-IDF 计算。

3. `min_df=0`

最小文档出现频次阈值：
`min_df` = minimum document frequency
含义：一个词汇**至少要出现在多少条文本里，才会被纳入词汇表当作特征**

- `min_df=0`：所有出现过的单词 / 短语全部保留，哪怕某个词组只在 1 条酒店描述里出现 1 次也不删除；

> 
> 日常实操一般设 `min_df=2/3`，过滤只出现一次的生僻小众短语，减少无用特征、降低维度；这里 0 代表不做过滤。

4. `stop_words='english'`

加载英文停用词词库，自动剔除无意义虚词：
the, a, an, and, is, in, on, at, to 这类介词、冠词、连词
这类词到处都出现，区分不了文本差异，TF-IDF 价值极低，提前删掉。

整体：构建了一个可以抽取 1~3 词短语、去除停用词、保留所有词汇的 TF-IDF 计算器。

## 文本转为 TF-IDF 稀疏特征矩阵
`df['desc_clean']`：清洗完毕的酒店英文描述文本列（去符号、小写、纠错后的干净文本）
`fit_transform()` = 两步合并：

1. **fit () 拟合学习**
遍历全部文本：分词、提取 1~3 元词组、过滤停用词，生成全局词汇字典，给每一个词组分配唯一列下标；同时统计全局文档信息，用来计算 IDF 权重。
2. **transform () 转换**
基于上面学到的词典，给每一段酒店描述计算 **TF-IDF 权重值**，最终生成稀疏矩阵：

- 行：每一行 = 1 条酒店描述
- 列：每一列 = 一个单词 / 短语特征
- 单元格数值：该词组在本条文本中的 TF-IDF 权重（0~1 之间小数）

### TF-IDF 简单对比词袋 (CountVectorizer)

之前 CountVectorizer 存的是**单词出现次数**；
TF-IDF 存的是**加权权重**：

- 高频通用词汇（比如 hotel）权重被压低
- 只在少数文本出现的特色词汇（spacious balcony 宽敞阳台）权重更高
作用：凸显每条酒店描述里独有的特色词汇，更适合后续聚类、分类、相似度计算。

### 存储形式

和词袋一样：**CSR 稀疏矩阵**，大量词汇在单条文本中权重为 0，只存储非 0 权重位置，节省内存。

In [46]:
df.set_index('name', inplace = True)

In [47]:
# 使用EF-IDF提取文本特征
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=0.0, stop_words='english') # min_df的值必须为一个浮点数
tfidf_matrix = tf.fit_transform(df['desc_clean'])

In [48]:
print(tfidf_matrix)

  (0, 13987)	0.03622101609530416
  (0, 22297)	0.06132962494208439
  (0, 24249)	0.06132962494208439
  (0, 13158)	0.0697422599700938
  (0, 24885)	0.07487044615101399
  (0, 10976)	0.04174074245572223
  (0, 9855)	0.042835930607239364
  (0, 12311)	0.02749572923998043
  (0, 20526)	0.02485864502103179
  (0, 6951)	0.018244904648252548
  (0, 11343)	0.015387186843180311
  (0, 17640)	0.04693440333435737
  (0, 3564)	0.0477831195009121
  (0, 13413)	0.03673859898309757
  (0, 16220)	0.031422694628590084
  (0, 11098)	0.027802168156432768
  (0, 16600)	0.06132962494208439
  (0, 14514)	0.03982116099484098
  (0, 12540)	0.031004802089972444
  (0, 5261)	0.04693440333435737
  (0, 12139)	0.028117003696100748
  (0, 1197)	0.0348711299850469
  (0, 10084)	0.04693440333435737
  (0, 15126)	0.050800731229431836
  (0, 9896)	0.0487057146301888
  :	:
  (151, 4955)	0.05206716402549704
  (151, 23942)	0.05206716402549704
  (151, 25962)	0.05206716402549704
  (151, 23697)	0.05206716402549704
  (151, 26277)	0.052067164025497

In [49]:
print(tfidf_matrix.shape)

(152, 26744)


tfidf_matrix：你上一步得到的 TF-IDF 稀疏特征矩阵

行数：酒店描述文本总数 N
每一行：一条酒店描述被转化成的数字向量

余弦相似度 (Cosine Similarity)：用来衡量两个文本向量之间有多相似，取值范围 [−1,1]

越接近 1：两段文字内容高度相似
等于 0：完全不相关、无共同点
负数：语义相反（文本场景很少出现）

![alt text](image.png)

![alt text](image-1.png)

![alt text](image-2.png)

In [50]:
# 计算酒店之间的余弦相似度（线性核函数）
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)
print(cosine_similarities)

[[1.         0.01406466 0.03391973 ... 0.01595671 0.00239758 0.00725341]
 [0.01406466 1.         0.02190086 ... 0.01636756 0.00431995 0.00896333]
 [0.03391973 0.02190086 1.         ... 0.02401214 0.00727375 0.01101182]
 ...
 [0.01595671 0.01636756 0.02401214 ... 1.         0.01264527 0.00870722]
 [0.00239758 0.00431995 0.00727375 ... 0.01264527 1.         0.00273408]
 [0.00725341 0.00896333 0.01101182 ... 0.00870722 0.00273408 1.        ]]


In [51]:
indices = pd.Series(df.index) #df.index是酒店名称

In [52]:
indices

0               Hilton Garden Seattle Downtown
1                       Sheraton Grand Seattle
2                Crowne Plaza Seattle Downtown
3                Kimpton Hotel Monaco Seattle 
4                           The Westin Seattle
                        ...                   
147                  The Halcyon Suite Du Jour
148                                Vermont Inn
149                 Stay Alfred on Wall Street
150         Pike's Place Lux Suites by Barsala
151    citizenM Seattle South Lake Union hotel
Name: name, Length: 152, dtype: object

In [53]:
# 基于相似度矩阵和指定的酒店name, 推荐TOP10酒店
def recommendations(name, cosine_similarities=cosine_similarities):
    recommended_hotels = []
    # 找到想要查询酒店名称的idx
    idx = indices[indices == name].index[0]
    print('idx=', idx)
    # 对于idx酒店的余弦相似度向量按照从大到小进行排序
    score_series = pd.Series(cosine_similarities[idx]).sort_values(ascending=False)
    # 取相似度最大的前十个（除自己之外）
    top_10_indexes = list(score_series.iloc[1:11].index)
    # 放到推荐列表中
    for i in top_10_indexes:
        recommended_hotels.append(list(df.index)[i])
    return recommended_hotels

In [54]:
print(recommendations('Hilton Seattle Airport & Conference Center'))
print(recommendations('The Bacon Mansion Bed and Breakfast'))

idx= 49
['DoubleTree by Hilton Hotel Seattle Airport', 'Embassy Suites by Hilton Seattle Tacoma International Airport', 'Four Points by Sheraton Seattle Airport South', 'Seattle Airport Marriott', 'Homewood Suites by Hilton Seattle-Tacoma Airport/Tukwila', 'Hampton Inn Seattle-Airport', 'Best Western Seattle Airport Hotel', 'Knights Inn Tukwila', 'Econo Lodge SeaTac Airport North', 'Motel 6 Seattle Sea-Tac Airport South']
idx= 116
['11th Avenue Inn Bed and Breakfast', 'Shafer Baillie Mansion Bed & Breakfast', 'Chittenden House Bed and Breakfast', 'Gaslight Inn', 'Bed and Breakfast Inn Seattle', 'Silver Cloud Hotel - Seattle Broadway', 'Hyatt House Seattle', 'Mozart Guest House', 'Holiday Inn Seattle Downtown', 'Quality Inn & Suites Seattle Center']
